# 🌲 03 — XGBoost Regressor
## Gradient Boosting avec validation croisée temporelle

Ce notebook utilise les features préparées par `00_Preprocessing` pour entraîner un XGBoost avec grid search et importance des features.

## 1. 📦 Imports & Chargement

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

print("✅ Imports OK")

In [ ]:
# Charger les données préparées
train = pd.read_csv("train_data.csv", index_col=0, parse_dates=True)
test = pd.read_csv("test_data.csv", index_col=0, parse_dates=True)

FEATURES = [c for c in train.columns if c != "revenue"]
TARGET = "revenue"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"✅ Données chargées")
print(f"   X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"   X_test : {X_test.shape}, y_test : {y_test.shape}")
print(f"   Features: {len(FEATURES)}")

## 2. ⚙️ Grid Search XGBoost avec CV Temporelle

In [ ]:
# Normalisation
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Grid Search
param_grid = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "subsample": 0.8},
    {"n_estimators": 200, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8},
    {"n_estimators": 150, "max_depth": 3, "learning_rate": 0.08, "subsample": 0.9},
    {"n_estimators": 300, "max_depth": 2, "learning_rate": 0.1, "subsample": 0.7},
]

# TimeSeriesSplit adapté à X_train
tscv = TimeSeriesSplit(n_splits=2, test_size=6)

print("⏳ Grid Search XGBoost...")
best_xgb, best_score = None, np.inf

for params in param_grid:
    model = XGBRegressor(**params, random_state=42, verbosity=0)
    model.fit(X_train_sc, y_train, verbose=False)
    
    val_preds, val_trues = [], []
    for train_idx, val_idx in tscv.split(X_train_sc):
        X_fold_train = X_train_sc[train_idx]
        y_fold_train = y_train.iloc[train_idx]
        X_fold_val = X_train_sc[val_idx]
        y_fold_val = y_train.iloc[val_idx]
        
        fold_model = XGBRegressor(**params, random_state=42, verbosity=0)
        fold_model.fit(X_fold_train, y_fold_train, verbose=False)
        val_preds.extend(fold_model.predict(X_fold_val))
        val_trues.extend(y_fold_val.values)
    
    cv_mape = np.mean(np.abs(np.array(val_trues) - np.array(val_preds)) / np.array(val_trues)) * 100
    print(f"   depth={params['max_depth']} n={params['n_estimators']} lr={params['learning_rate']} → CV MAPE: {cv_mape:.2f}%")
    
    if cv_mape < best_score:
        best_score = cv_mape
        best_xgb = model

# Prédictions
xgb_pred = best_xgb.predict(X_test_sc)
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
mape_val = np.mean(np.abs((y_test.values - xgb_pred) / y_test.values)) * 100
r2 = r2_score(y_test, xgb_pred)

print(f"\n📊 XGBoost (grid search)")
print(f"   MAE  : {mae:>12,.0f} €")
print(f"   RMSE : {rmse:>12,.0f} €")
print(f"   MAPE : {mape_val:>11.2f} %")
print(f"   R²   : {r2:>11.4f}")

## 3. 📊 Feature Importance & Visualisation

In [ ]:
# Importance des features
importances = pd.Series(best_xgb.feature_importances_, index=FEATURES).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
y_train.plot(ax=ax, label="Train", color="#1565C0", alpha=0.6, linewidth=2)
y_test.plot(ax=ax, label="Test réel", color="#2E7D32", linewidth=2.5)
pd.Series(xgb_pred, index=y_test.index).plot(ax=ax, label="XGBoost", color="#6A1B9A", linestyle="--", linewidth=2.5)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-30"), alpha=0.07, color="red", label="COVID")
ax.set_title("XGBoost — Prévisions vs Réel", fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M€"))
ax.legend(); ax.grid(alpha=0.3)

ax2 = axes[1]
top_features = importances.head(12)
colors_feat = ["#1B5E20" if v > 0.03 else "#90A4AE" for v in top_features.values]
top_features.plot(kind="barh", ax=ax2, color=colors_feat)
ax2.set_title("Top 12 Features XGBoost", fontsize=12, fontweight="bold")
ax2.set_xlabel("Importance"); ax2.grid(alpha=0.3, axis="x")

plt.suptitle(f"XGBoost | MAPE: {mape_val:.2f}% | R²: {r2:.4f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("modele3_xgboost.png", dpi=120, bbox_inches="tight")
plt.show()
print("💾 Graphique → modele3_xgboost.png")

print(f"\n📌 Top 5 features :")
for feat, imp in importances.head(5).items():
    print(f"   {feat:>20} : {imp:.3f}")

# Sauvegarder
results_df = pd.DataFrame({"date": y_test.index, "reel": y_test, "prediction": xgb_pred})
results_df.to_csv("predictions_xgboost.csv", index=False)
print("💾 Prédictions → predictions_xgboost.csv")